# PanVision — preprocess a balanced subset (run before the training notebooks)

Uses the team's preprocessing code unchanged (`src/preprocessing`): HU window −100..300 → 0..1, 1.5×1.5×3 mm,
LPS. Each CT is downloaded from the public HuggingFace mirror to Colab's local disk, processed, copied to
`processed/full/<study_id>/` in the shared Drive folder (~10 MB each), and the raw CT (~70 MB) is deleted.

**Runtime → Change runtime type → CPU** (no GPU needed; keeps your GPU quota for training).
If the session disconnects, run all cells again: finished studies are skipped.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get('GITHUB_TOKEN')
!rm -rf /content/Panvision && git clone -q -b segmentation https://{token}@github.com/hrihaanG/Panvision.git /content/Panvision
!git -C /content/Panvision remote set-url origin https://github.com/hrihaanG/Panvision.git
%cd /content/Panvision
!pip -q install SimpleITK openpyxl huggingface_hub

In [ ]:
import os, glob
hits = sorted(set(p[:-len('/raw/annotations')] for p in
                  glob.glob('/content/drive/MyDrive/**/raw/annotations', recursive=True)))
print('\n'.join(hits) or 'Not found: add the shared folder as a shortcut in My Drive.')
if not os.path.exists('/content/data'):
    os.symlink(hits[0], '/content/data')  # the shared folder name has spaces; use a space-free link
print('Using', os.path.realpath('/content/data'))
# Official annotations (the Drive copy lacks manual_labels/): public, ~80 MB.
!rm -rf /content/panorama_labels && git clone -q --depth 1 https://github.com/DIAGNijmegen/panorama_labels.git /content/panorama_labels
!ls /content/panorama_labels

## Plan (no downloads)
Per split: half PDAC with a **manual** lesion mask, half non-PDAC. ~10 MB per study in Drive
(460 studies ≈ 4.6 GB, counted against the Drive of whoever runs this). Increase `--train` if you have space and time.

In [ ]:
!python -m src.segmentation.prepare_subset --shared /content/data --annotations /content/panorama_labels --train 300 --val 80 --test 80 --dry-run

## Run (~1–2 h; re-run after a disconnect to continue)

In [ ]:
!python -m src.segmentation.prepare_subset --shared /content/data --annotations /content/panorama_labels --train 300 --val 80 --test 80